# M1 — arXiv crawler (Team Colab)

**Quy trình:** GitHub branch → Mount Drive → Cấu hình range → Fast Crawl → Kiểm tra checkpoint → Recovery theo phiên bản → Export report.

- Dùng **Runtime → Change runtime type → CPU**.
- Không chạy song song nhiều notebook cùng truy cập arXiv.
- Mỗi người dùng thư mục Drive riêng và **giữ nguyên `BATCH_SIZE` khi resume**.
- Code trong repo cần có phiên bản `run_arxiv_only.py` đã sửa Recovery và `export_reports.py`.


## 1 — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/OWNER/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "feature/arxiv-crawler"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/m1_crawling/requirements-crawler.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_arxiv_crawler.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)


## 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3 — Chọn khoảng ID và thư mục lưu

In [ ]:
RANGE_ID = 5             # 1..5, mỗi thành viên chọn range được giao
BATCH_SIZE = 50          # giữ nguyên khi chạy lại / Recovery
DATA_DIR = Path(f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Range:", RANGE_ID, "| Batch size:", BATCH_SIZE, "| Drive:", DATA_DIR)


## 4 — Fast Crawl: không retry, tự lưu checkpoint

`batches=1` là một batch 50 ID. Gọi lại hàm để chạy các batch tiếp theo. Nếu arXiv báo giới hạn truy cập, dừng và chờ thay vì tăng tốc/đa luồng.

In [ ]:
def _run_crawler(extra_args):
    cmd = [sys.executable, "-m", "src.m1_crawling.run_arxiv_only",
           "--range", str(RANGE_ID),
           "--batch-size", str(BATCH_SIZE),
           "--backup-dir", str(DATA_DIR),
           "--delay", "3.1", *extra_args]
    result = subprocess.run(cmd, cwd=PROJECT)
    # Exit code 2 means some source downloads failed, but the batch was saved.
    if result.returncode not in (0, 2):
        raise RuntimeError(f"Crawler failed with exit code {result.returncode}")
    return result.returncode

def fast_crawl(batches=1):
    return _run_crawler(["--next", str(batches), "--retries", "0"])

fast_crawl(batches=1)


## 5 — Kiểm tra các checkpoint đã lưu

Việc lưu source + metadata xảy ra **trong lúc crawl**, không phải đợi đến cell này.

In [ ]:
import json

def show_checkpoints():
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Đã lưu {len(manifests)} checkpoint")
    for marker in manifests:
        info = json.loads(marker.read_text(encoding="utf-8"))
        archive = marker.with_suffix(".tar.gz")
        if not archive.exists():
            print("MISSING ARCHIVE:", marker.name)
            continue
        print(f"{marker.stem}: ok={info.get('ok')}, error={info.get('failed')}, "
              f"no_tex={info.get('no_tex')}, size={archive.stat().st_size/1024**2:.2f} MiB")

show_checkpoints()


## 6 — Recovery: chỉ tải lại các phiên bản lỗi

Hàm này **sao lưu các checkpoint lỗi trước lần Recovery đầu tiên**. Crawler được sửa để giữ nguyên `v1` thành công và chỉ tải lại `v2` lỗi (nếu có). `no_tex` không bị retry.

In [ ]:
import shutil

def recovery_crawl(batches=1, retries=0, backup_before=True):
    if backup_before:
        snapshot = DATA_DIR / "_before_recovery"
        snapshot.mkdir(parents=True, exist_ok=True)
        copied = 0
        for marker in sorted(DATA_DIR.glob("batch_*.json")):
            info = json.loads(marker.read_text(encoding="utf-8"))
            if int(info.get("failed", 0)) <= 0:
                continue
            archive = marker.with_suffix(".tar.gz")
            if not archive.exists():
                raise FileNotFoundError(f"Missing archive: {archive}")
            for source in (marker, archive):
                destination = snapshot / source.name
                if not destination.exists():
                    shutil.copy2(source, destination)
                    copied += 1
        print(f"Saved {copied} original checkpoint files in {snapshot}")
    return _run_crawler(["--next", str(batches), "--retry-failed",
                         "--retries", str(retries)])

# Chạy khi cần Recovery. Lần đầu nên thử 1 batch.
recovery_crawl(batches=1, retries=0)


## 7 — Export report: CSV, JSON, danh sách phiên bản lỗi

`metrics.csv` là số liệu **lần thực thi cuối của mỗi batch**, không cộng dồn Fast + Recovery.

In [ ]:
from src.m1_crawling.export_reports import export_reports


def export_report(verify_sha256=False):
    summary = export_reports(DATA_DIR, verify_sha256=verify_sha256)
    print("===== REPORT =====")
    for key in ("batch_count", "total_records", "metadata_rows", "ok", "error",
                "no_tex", "not_found", "failed_version_rows",
                "latest_checkpoint_runtime_seconds"):
        print(f"{key}: {summary[key]}")
    print("Saved at:", DATA_DIR / "reports")
    for path in sorted((DATA_DIR / "reports").iterdir()):
        print(" -", path.name)
    return summary

summary = export_report()


## Sau khi chạy

- Gửi người xử lý dữ liệu: `reports/papers.csv`, `reports/crawl_records.csv`, **archive `batch_*.tar.gz`** và đường dẫn Drive.
- `reports/failed_versions.csv` cho biết phiên bản nào còn lỗi.
- `no_tex` là một trạng thái dữ liệu, không đồng nghĩa với lỗi mạng.
- Nếu muốn so sánh Fast với Recovery, giữ `reports/` của Fast Pass ở một thư mục snapshot **trước khi** xuất report mới.
